
# 03 — Causal Self-Attention & the KV Cache, from Scratch

**Goal:** the "can't fail this" foundations round. Implement causal multi-head self-attention
with an incremental KV cache, verify step-by-step decoding matches a full forward pass exactly,
and be fluent on MHA vs. GQA vs. MQA and why KV cache size is a first-order systems concern (the
thing that makes speculative decoding, quantization, and architecture choices like GQA all
connect back to the same underlying bottleneck).

Structure: **Lesson → Implementation → Quiz → Final Answers & Explanations.**



## 1. Lesson

### 1.1 Self-attention recap

For a single sequence of length $L$ with embeddings $X \in \mathbb{R}^{L \times d}$:

$$Q = XW_Q,\quad K = XW_K,\quad V = XW_V$$
$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V$$

Multi-head: split $d$ into $h$ heads of size $d_k = d/h$, run this independently per head,
concatenate outputs, apply $W_O$. Same idea as cross-attention (notebook 1) — here $Q, K, V$ all
come from the *same* sequence.

### 1.2 Causal masking

For autoregressive language modeling, token $i$ must not attend to token $j > i$ (it hasn't been
generated yet, and allowing it would leak future information both at train and inference time).
$M$ is set to $-\infty$ for all $(i,j)$ with $j > i$, zero elsewhere, added to scores before
softmax — after softmax those positions get exactly zero weight.

### 1.3 Why a KV cache exists

Naively, generating token $L{+}1$ from scratch means re-running the full forward pass over all
$L{+}1$ tokens — recomputing $K, V$ (and the attention output) for tokens $1, \dots, L$ *again*,
even though nothing about their $K,V$ projections changed (they only depend on tokens up to their
own position, and those are fixed once generated). The **KV cache** stores $K$ and $V$ for every
past position after they're computed once; at each new decode step you only need to compute $Q,
K, V$ for the **single new token**, append the new $K,V$ to the cache, and attend the new $Q$
against the *entire* cached $K,V$ (old + new). This turns each decode step's attention cost from
$O(L^2 d)$ (recompute everything) down to $O(L d)$ (attend once against a cache of size $L$) —
linear instead of quadratic per step, at the cost of **memory**: the cache must be stored,
per-layer, per-head, for the whole sequence so far.

### 1.4 KV cache size — the memory cost

For a single layer, the cache holds $K, V \in \mathbb{R}^{\text{batch} \times \text{heads} \times L \times d_k}$
each. Total cache memory across all layers scales as
$$
2 \times \text{num\_layers} \times \text{batch} \times \text{heads} \times L \times d_k \times \text{bytes\_per\_elt}
$$
This grows *linearly* with sequence length $L$ and *linearly* with batch size — for long
contexts and large batches, KV cache can dwarf the model weights themselves in memory footprint,
directly limiting how long a context or how large a batch you can serve. This is precisely the
motivation for:

- **Multi-Query Attention (MQA)**: use $h$ query heads but only **1** shared key/value head (all
  query heads attend against the same $K,V$). Cache shrinks by a factor of $h$. Cost: less
  representational diversity in what different heads can attend to, since they all share the
  same $K,V$ projection.
- **Grouped-Query Attention (GQA)**: a middle ground — group the $h$ query heads into $g$ groups
  (with $1 < g < h$), each group sharing one $K,V$ head. Cache shrinks by a factor of $h/g$
  instead of $h$. This is what most modern open LLMs (Llama 2/3 70B, Qwen, Mistral, etc.) actually
  use in practice — it recovers most of MQA's memory savings while keeping noticeably more of
  MHA's quality, since a handful of *groups* of heads still specialize somewhat differently.

Interview framing: MHA = best quality / largest cache, MQA = smallest cache / some quality loss,
GQA = tunable point on that trade-off curve, and it's the one actually deployed almost
everywhere today.

### 1.5 Prefill vs. decode, and why causal masking still matters with a cache

Generation happens in two phases: **prefill** (the initial prompt, all $L_0$ tokens processed in
one parallel forward pass — here you *do* need the full causal mask, since all prompt tokens are
present at once but shouldn't see "future" prompt tokens) and **decode** (one new token per step,
appended to the cache — here there's only ever *one* new query row per step, but it must attend
to *all* cached past positions and nothing after itself, which is automatically satisfied since
nothing after it exists yet in the cache). So causal masking during prefill is essential (a
full $L_0 \times L_0$ mask); during decode, the "mask" is trivial (the single new query always
sees all — and only — cached past positions), but the *invariant* being preserved is identical:
no position ever attends to a position generated after it.

### 1.6 Connection to the rest of the interview

- KV cache size directly determines max servable batch size / context length — a systems
  question that shows up constantly ("why can't we just increase the batch size?").
  GQA/MQA, and cache quantization, are the standard answers.
- Speculative decoding (notebook 2) needs the *target* model's KV cache extended by exactly the
  drafted+accepted tokens each round, and any rejected tail must be **rolled back** (cache
  entries for discarded speculative positions must be dropped) — a detail worth mentioning if
  asked how speculative decoding interacts with KV caching.
- Rotary Position Embeddings (RoPE), used in most modern LLMs (Llama, Qwen, etc.), are applied to
  $Q$ and $K$ (as a rotation depending on absolute position) but *not* to $V$ — because RoPE's
  purpose is to make the $Q \cdot K$ dot product a function of *relative* position only; $V$ is
  just the content being mixed, it has no notion of "position" to encode, so rotating it would
  serve no purpose (and would need to be un-rotated somewhere to be meaningful, which nothing
  does). We don't implement RoPE below, but expect this exact question.


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)



## 2. Implementation

Implement `CausalSelfAttention` supporting an optional incremental KV cache. Fill in every
`# TODO`.

- `forward(x, cache=None)`
  - `x`: `(batch, L_new, embed_dim)` — the **new** tokens only (during decode, `L_new == 1`;
    during prefill, `L_new == L_0`, the whole prompt).
  - `cache`: `None` (no cache yet) or a dict `{'k': (batch, heads, L_past, head_dim), 'v': (...)}`
    holding everything computed so far.
  - Returns `(output, new_cache)` where `output` is `(batch, L_new, embed_dim)` and `new_cache`
    is the cache dict extended with the new tokens' K, V.
  - Causal masking must be correct for **both** cases: when `cache is None` (need a full causal
    mask over the `L_new x L_new` block) and when `cache` has past entries (the new queries must
    see all past + all new-so-far-including-themselves, but nothing later than themselves among
    the new tokens — i.e. still causal *within* the new block, but unrestricted against the past).


In [ ]:

class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        # TODO: qkv_proj as one combined nn.Linear(embed_dim, 3*embed_dim) is fine, or three
        # separate ones -- your choice. Also define self.out_proj.
        self.qkv_proj = None
        self.out_proj = None

    def _split_heads(self, x):
        b, s, _ = x.shape
        # TODO: (batch, seq, embed_dim) -> (batch, heads, seq, head_dim)
        raise NotImplementedError

    def forward(self, x: torch.Tensor, cache: dict = None):
        batch, L_new, _ = x.shape

        # TODO: project x -> Q, K, V for the NEW tokens only, each (batch, L_new, embed_dim),
        # then split into heads -> (batch, heads, L_new, head_dim)
        Q = None
        K_new = None
        V_new = None

        if cache is not None:
            # TODO: concatenate cache['k'] (past) with K_new along the sequence dim (dim=2),
            # same for V. This gives the FULL K, V (past + new) to attend against.
            K = None
            V = None
        else:
            K = K_new
            V = V_new

        L_past = K.shape[2] - L_new  # how many cached (past) positions existed before this call

        # TODO: scores = Q @ K^T / sqrt(head_dim), shape (batch, heads, L_new, L_past + L_new)
        scores = None

        # TODO: build the causal mask for this block. Query row i (0-indexed within the NEW
        # block) is allowed to see: ALL of the L_past cached positions, PLUS new positions
        # 0..i (inclusive) within the new block, but NOT new positions > i.
        # Hint: build a (L_new, L_past + L_new) boolean mask where True = "not allowed" (to be
        # filled with -inf), then broadcast to (1, 1, L_new, L_past+L_new) before applying.
        mask = None
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        # TODO: softmax + weighted sum with V -> (batch, heads, L_new, head_dim)
        attn_weights = None
        out = None

        # TODO: merge heads -> (batch, L_new, embed_dim), apply out_proj
        out = None

        new_cache = {"k": K, "v": V}
        return out, new_cache



### Sanity tests

1. **Shape check** on a plain prefill call (no cache).
2. **Correctness of causal masking**: a full no-cache forward pass over a whole sequence must
   produce, at each position $i$, exactly the same output as if you'd only been allowed to see
   tokens $0..i$ — i.e. changing tokens *after* position $i$ must not change position $i$'s
   output (this is what causal masking guarantees, and it's easy to silently break).
3. **The big one — incremental decoding matches full prefill exactly**: feed a sequence token by
   token through the cache, one at a time, and confirm the concatenation of all per-step outputs
   exactly matches a single full forward pass with no cache at all. This is the property that
   makes KV caching a pure speed optimization with **zero** change in output — if your cache
   implementation is correct, decode-with-cache and decode-without-cache are numerically
   identical, not just "close."


In [ ]:

embed_dim, num_heads = 32, 4
batch, L = 2, 6

attn = CausalSelfAttention(embed_dim, num_heads)
x = torch.randn(batch, L, embed_dim)

out, cache = attn(x)
assert out.shape == (batch, L, embed_dim)
assert cache["k"].shape == (batch, num_heads, L, embed_dim // num_heads)
print("Test 1 passed: shapes correct,", out.shape, cache["k"].shape)


In [ ]:

# Test 2: causal masking -- perturbing tokens AFTER position i must not change position i's output.
attn2 = CausalSelfAttention(embed_dim, num_heads)
x2 = torch.randn(1, L, embed_dim)
out2, _ = attn2(x2)

x2_perturbed = x2.clone()
x2_perturbed[:, 3:, :] = torch.randn(1, L - 3, embed_dim)  # change everything from position 3 onward
out2_perturbed, _ = attn2(x2_perturbed)

# positions 0,1,2 must be untouched; positions 3+ are expected to differ
assert torch.allclose(out2[:, :3], out2_perturbed[:, :3], atol=1e-5), \
    "causal mask is leaking future information into earlier positions!"
assert not torch.allclose(out2[:, 3:], out2_perturbed[:, 3:], atol=1e-5), \
    "sanity check: perturbed positions should actually differ (test setup issue if this fires)"
print("Test 2 passed: no future-information leakage through the causal mask")


In [ ]:

# Test 3: incremental decode-with-cache == full prefill, exactly.
attn3 = CausalSelfAttention(embed_dim, num_heads)
x3 = torch.randn(1, L, embed_dim)

full_out, _ = attn3(x3)  # single forward pass, no cache

step_outputs = []
cache = None
for t in range(L):
    tok = x3[:, t:t+1, :]  # one new token
    step_out, cache = attn3(tok, cache=cache)
    step_outputs.append(step_out)
incremental_out = torch.cat(step_outputs, dim=1)

max_diff = (full_out - incremental_out).abs().max().item()
assert torch.allclose(full_out, incremental_out, atol=1e-5), f"mismatch, max diff={max_diff}"
print(f"Test 3 passed: incremental cached decode exactly matches full prefill (max diff {max_diff:.2e})")



## 3. Quiz

1. Why is a KV cache a memory/compute trade-off rather than a "free" optimization — what,
   precisely, do you spend more of, and what do you save?
2. Give the KV cache's memory footprint formula (in terms of layers, batch, heads, sequence
   length, head dim, dtype size) and explain which of these terms is the one that makes very
   long contexts or very large batch sizes expensive to serve.
3. Explain MHA vs. GQA vs. MQA. Why has GQA become the default in most modern open-weight LLMs
   rather than plain MHA or full MQA?
4. Why does causal masking need to cover the *entire* $L_0 \times L_0$ block during prefill, but
   reduce to something close to "trivial" during single-token decode steps? What invariant is
   preserved identically in both cases?
5. Why is a KV cache *exactly* mathematically equivalent to recomputing everything from scratch
   at each step (not an approximation)? What implementation mistake would break that exact
   equivalence?
6. RoPE is applied to $Q$ and $K$ but never to $V$. Why not?
7. How does a growing KV cache interact with speculative decoding's rollback step (discarding
   rejected drafted tokens)? What has to happen to the cache when a drafted continuation is
   rejected partway through?

*(Your answers here)*



## 4. Final Answers & Explanations

### Q1 — The trade-off
Without a cache, generating token $t{+}1$ recomputes $K,V$ (and full attention) for *all* $t$
prior tokens from scratch every single step — $O(L^2 d)$ total compute across a full generation
of length $L$ just for attention, dominated by pure waste (recomputing the exact same $K,V$
values over and over). A KV cache spends **memory** (storing every past position's $K,V$
per layer, per head) to avoid that **recomputation**, reducing each step's attention cost to
$O(Ld)$ (attend once against the cache) — linear per step, $O(L^2d)$ total across the whole
generation instead of $O(L^3d)$-ish redundant work without it. So: you trade memory (which is
finite and often the binding constraint on GPUs) for compute you'd otherwise waste identically
recomputing.

### Q2 — Memory formula
$$
\text{cache bytes} = 2 \times \text{num\_layers} \times \text{batch} \times \text{num\_kv\_heads} \times L \times d_{head} \times \text{bytes\_per\_element}
$$
(the factor of 2 is for storing both $K$ and $V$). The term that makes this dangerous in
practice is $L$ (sequence length) combined with **batch size**: both multiply the cache
linearly, so serving long contexts *and* large batches simultaneously multiplies memory
demand in both dimensions at once — this is exactly why long-context serving and high-throughput
batched serving compete for the same memory budget, and why cache-size-reducing techniques
(GQA/MQA, quantized KV cache, cache eviction/windowing) are load-bearing in production LLM
serving, not academic curiosities.

### Q3 — MHA vs GQA vs MQA
**MHA** (multi-head attention): each of the $h$ query heads has its *own* $K,V$ projection —
maximum representational flexibility (different heads can specialize in attending to very
different things), but cache size scales with the full $h$. **MQA** (multi-query): all $h$ query
heads share a single $K,V$ head — cache shrinks by a factor of $h$, but all heads are now forced
to "look through the same lens" for keys/values, costing some quality/expressiveness. **GQA**
(grouped-query): partition the $h$ query heads into $g$ groups ($1 < g < h$), each group sharing
one $K,V$ head — cache shrinks by $h/g$ instead of $h$, a tunable dial between MHA and MQA. GQA
won out in practice (Llama 2/3, Qwen, Mistral, etc.) because empirically it recovers nearly all
of MHA's quality (a handful of head-groups is enough diversity for most of what heads actually
specialize in) while capturing most of MQA's cache savings — it dominates the quality/memory
trade-off curve at the group counts people actually use (e.g. 8 KV heads for 32-64 query heads).

### Q4 — Prefill vs decode masking
During prefill, all $L_0$ prompt tokens are processed *simultaneously* in one forward pass, so
without an explicit mask, every token could see every other token including ones "after" it in
the sequence — violating autoregressive validity, so the full lower-triangular
$L_0\times L_0$ mask is mandatory. During decode, there's only ever *one* new query row per step,
and everything in the cache (by construction) was generated *before* this step — so "don't
attend to the future" is automatically satisfied: there simply is no future content in the cache
yet to accidentally attend to. The invariant preserved identically in both regimes is: **no
position ever attends to a position that didn't exist yet at the time it was computed.** Prefill
needs an explicit mask to enforce this within a batch of simultaneous positions; decode gets it
for free because of the strict step-by-step order the cache is built in.

### Q5 — Exact equivalence, and what breaks it
KV caching is exact (not approximate) because $K_i, V_i$ for a given position $i$ depend *only*
on token $i$'s own embedding (and that layer's weights) — never on any *later* token — so once
computed, $K_i,V_i$ are valid forever and identical to what a full recomputation would produce.
Caching just avoids recomputing values that provably cannot have changed. What would break this:
(a) using a non-causal / bidirectional variant where later tokens influence earlier ones' K/V
(then caching would be invalid — you couldn't cache B, since a future token might change it);
(b) any per-position transformation that depends on full-sequence context at the *projection*
step itself, before caching (rare, but e.g. certain normalization schemes computed over the
whole sequence rather than per-token would break this); (c) simply an off-by-one bug in the mask
that lets new queries see "future" new-block positions they shouldn't — that wouldn't break the
cache mechanism per se, but would silently make cached and uncached outputs disagree, exactly
what Test 3 above is designed to catch.

### Q6 — Why not rotate V
RoPE encodes *position* by rotating $Q$ and $K$ vectors by an angle proportional to their
absolute position, engineered so that the dot product $Q_i \cdot K_j$ after rotation depends only
on the *relative* offset $(i-j)$, not on $i$ and $j$ individually — that's the entire mechanism
by which RoPE injects positional information into attention *scores*. $V$ plays a completely
different role: it's the content that gets mixed together according to whatever attention
weights the (already position-aware) scores produced — it has no further need to "know" its own
position, since position has already done its job in shaping the attention weights by the time
$V$ is used. Rotating $V$ would just be applying an arbitrary, purposeless transformation to the
content being aggregated, with no mechanism to ever undo it meaningfully — it isn't part of how
positional information is supposed to flow through the layer.

### Q7 — Cache rollback under speculative rejection
The target model's KV cache is extended, during verification, with entries for **all** $k$
drafted positions at once (since verification processes them all in one parallel pass — see
notebook 2, Q7). But if position $i$ is rejected, only positions $1, \dots, i{-}1$ (accepted)
plus the one replacement/residual token actually belong in the "real" generated sequence — the
cached $K,V$ for drafted positions $i, i{+}1, \dots, k$ (the rejected one and everything
after it, which was conditioned on a draft path that's now being discarded) must be **evicted**
from the cache before the next round begins, otherwise subsequent generation would be attending
to phantom, never-actually-generated tokens. Concretely: the cache is truncated back to length
(original length) + (number actually accepted) + 1 (for the replacement/bonus token), discarding
everything appended for positions beyond that. This rollback bookkeeping is one of the more
fiddly implementation details in real speculative decoding systems.
